# Multi-Agent Dino Runner Game — Full Pipeline (with QA Subgraph)

This is the complete, merged implementation: the main 8-node pipeline
(Director → Architect → Engineer → Save Code → Execution → QA → Scorer →
Human Review) with the QA + Scorer step replaced by the Advanced
Extension's **QA Subgraph** — a compiled 3-node subgraph
(`syntax_checker` → `logic_tester` → `performance_auditor`) that appears
to the parent graph as a single `"qa"` node.

In [0]:
%pip install langgraph langchain langchain-community langsmith langchain-groq pygame

Looking in indexes: [REDACTED]
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
dbutils.library.restartPython()

### Imports

In [0]:
import os
import re
import ast
import uuid
import operator
import subprocess
from typing import TypedDict, Annotated
from langgraph.graph.message import add_messages
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage
from langchain_groq import ChatGroq
from langgraph.types import interrupt, Command
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver

In [0]:
os.environ["GROQ_API_KEY"] = dbutils.secrets.get(scope="dino-runner-secrets", key="groq-api-key")
os.environ["LANGCHAIN_API_KEY"] = dbutils.secrets.get(scope="dino-runner-secrets", key="langsmith-api-key")
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = "dino-runner-multiagent"

llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0.2)

### Handling Provider-Specific Message Formats

Different LLM providers structure their response content differently -
OpenAI returns plain strings, while Gemini returns a list of content
blocks. A shared `get_text()` helper normalizes this across the pipeline,
so every node can reliably extract plain text regardless of which
provider generated it.

In [0]:
def get_text(message) -> str:
    """
    Extracts plain text from a message's .content, handling both
    OpenAI-style (content is a string) and Gemini-style (content is a
    list of dicts like [{'type': 'text', 'text': '...'}]) formats.
    """
    content = message.content
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        return "".join(
            part.get("text", "") for part in content if isinstance(part, dict)
        )
    return str(content)

## System Memory — Defining GameState

The shared memory structure passed from agent to agent. Fields that
accumulate history use reducers so new values are appended rather than
overwriting what came before.

In [0]:
class GameState(TypedDict):
    director_messages: Annotated[list, add_messages]
    architect_messages: Annotated[list, add_messages]
    engineer_code: Annotated[list, add_messages]
    qa_feedback: Annotated[list, add_messages]
    current_actor: str
    iteration: int
    iteration_score: Annotated[list[int], operator.add]
    file_saved: bool

## Main Pipeline Nodes

### Director Node
Requires no LLM call. Packages the initial objective as a message. The
exact required features are spelled out explicitly here so every
downstream agent — Architect, Engineer, QA — is grounded in the same
concrete checklist instead of a vague "build a dino game".

In [0]:
def director_node(state: GameState):
    prompt = (
        "Build a Dino Runner game (Chrome Dino style) in Python using pygame. "
        "It must include ALL of the following, exactly:\n"
        "1. Flying obstacles (e.g. Pterodactyls) at variable heights.\n"
        "2. Ground obstacles (e.g. Cacti).\n"
        "3. Accurate jumping and falling physics (gravity, velocity, a landing check).\n"
        "4. Jumping mapped to one key and ducking mapped to a different key, "
        "with a visibly smaller hitbox/sprite while ducking.\n"
        "5. A functional high-score tracking system (persists the best run's "
        "score within the session and displays both current score and high score)."
    )
    return {
        "director_messages": [HumanMessage(content=prompt)],
        "current_actor": "architect"
    }

### Architect Node
Reads the Director's request and produces a high-level system design —
the blueprint the Engineer will translate into actual code.

In [0]:
def architect_node(state: GameState):
    director_request = state["director_messages"][-1].content

    system_prompt = (
        "You are a software architect. Break down the following game request "
        "into a clear list of system components and features needed "
        "(e.g. Player class, obstacle spawner, collision detection, scoring system)."
    )

    response = llm.invoke([
        ("system", system_prompt),
        ("human", director_request)
    ])

    return {
        "architect_messages": [AIMessage(content=response.content)],
        "current_actor": "engineer"
    }

### Engineer Node
On the first pass, builds the game from the Architect's design. On every
pass after that, reads the previous code together with QA feedback and
returns a corrected full version — the node that actually improves with
each loop of the pipeline.

In [0]:
def strip_code_fences(text: str) -> str:
    """
    Some models (notably Llama via Groq) ignore "no markdown fences"
    instructions and wrap code in ```python ... ``` anyway. This strips
    a leading/trailing fence if present, so downstream ast.parse() and
    file execution see clean Python instead of failing on line 1.
    """
    text = text.strip()
    if text.startswith("```"):
        # drop the opening fence line (``` or ```python)
        text = text.split("\n", 1)[1] if "\n" in text else ""
    if text.endswith("```"):
        text = text.rsplit("```", 1)[0]
    return text.strip()

In [0]:
def engineer_node(state: GameState):
    design = get_text(state["architect_messages"][-1])

    if state["qa_feedback"]:
        latest_feedback = get_text(state["qa_feedback"][-1])
        instruction = (
            f"Here is the current game code and QA feedback on it. "
            f"Make ONLY the minimal changes needed to fix the specific issues "
            f"listed. Do NOT rewrite unrelated parts of the code, change the "
            f"art style, rename classes, or restructure the file. Preserve "
            f"everything that isn't broken. Return the FULL file with your "
            f"targeted fixes applied.\n\n"
            f"Current code:\n{get_text(state['engineer_code'][-1])}\n\n"
            f"QA Feedback:\n{latest_feedback}"
        )
    else:
        instruction = f"Write a complete Python Dino Runner game based on this design:\n{design}"

    system_prompt = (
        "You are a Python game developer using pygame. "
        "Always return ONLY the complete, runnable Python code — "
        "no explanations, no markdown code fences, just raw code."
    )

    response = llm.invoke([
        ("system", system_prompt),
        ("human", instruction)
    ])

    return {
        "engineer_code": [AIMessage(content=strip_code_fences(get_text(response)))],
        "current_actor": "execution_manager",
        "iteration": state["iteration"] + 1
    }

### File I/O and Execution Nodes
Saves code to disk, then runs it as a subprocess to see if it works. A
human-in-the-loop checkpoint sits before execution since running
AI-generated code automatically carries risk.

The execution node also sets `SDL_VIDEODRIVER=dummy` / `SDL_AUDIODRIVER=dummy`
on the subprocess's environment — headless environments (e.g. Databricks)
have no real display server, so without this, pygame's `display.set_mode()`
would crash immediately (not time out), which would look like a real bug
to QA even though the code is fine.

In [0]:
def save_code_node(state: GameState):
    latest_code = get_text(state["engineer_code"][-1])
    with open("dino_runner.py", "w") as f:
        f.write(latest_code)
    return {
        "file_saved": True,
        "current_actor": "execution_manager"
    }


def execution_node(state: GameState):
    """
    Pauses for human approval, then runs the saved code as a subprocess
    for a short window. A real game loop never exits on its own, so an
    8-second timeout with no crash is treated as a SUCCESS signal (code
    launched and is running healthily), not a failure - only an actual
    exception/traceback counts as something QA needs to flag.
    """
    approval = interrupt(
        {"question": "Approve running the generated code?(y/n)", "code_preview": get_text(state["engineer_code"][-1])[:300]}
    )

    if approval != "y":
        return {
            "qa_feedback": [ToolMessage(content="Execution was rejected by human reviewer.", tool_call_id="execution_manager")],
            "current_actor": "qa"
        }

    headless_env = dict(os.environ)
    headless_env["SDL_VIDEODRIVER"] = "dummy"
    headless_env["SDL_AUDIODRIVER"] = "dummy"

    try:
        result = subprocess.run(
            ["python", "dino_runner.py"],
            capture_output=True,
            text=True,
            timeout=8,
            env=headless_env
        )
        execution_log = f"STDOUT:\n{result.stdout}\n\nSTDERR:\n{result.stderr}"

    except subprocess.TimeoutExpired as e:
        partial_output = (e.stdout or b"").decode() if e.stdout else ""
        partial_errors = (e.stderr or b"").decode() if e.stderr else ""
        execution_log = (
            f"Process ran for 8+ seconds without crashing (timed out as expected "
            f"for a headless game loop with no window to close).\n\n"
            f"Partial STDOUT:\n{partial_output}\n\nPartial STDERR:\n{partial_errors}"
        )

    return {
        "qa_feedback": [ToolMessage(content=execution_log, tool_call_id="execution_manager")],
        "current_actor": "qa"
    }

## QA Subgraph (Advanced Extension)

Instead of a single flat `qa_node` + `scorer_node`, QA is its own
internal 3-node pipeline, compiled separately and plugged into the main
graph as one single node called `"qa"`.

- **`syntax_checker_node`** — no LLM. `ast.parse()` catches broken code
  instantly and for free.
- **`logic_tester_node`** — LLM. Only runs if syntax passed. Diagnoses
  functional bugs / missing features against the Director's original
  requirements and the Architect's design.
- **`performance_auditor_node`** — the scorer. Short-circuits to a score
  of 1 (no LLM call) if syntax failed; otherwise scores the logic report
  1–10 using the same banded rubric as before.

In [0]:
class QAState(TypedDict):
    # Same field names as GameState - LangGraph matches these
    # automatically when the compiled subgraph is added as a node,
    # so values pass through in both directions with no manual wiring.
    engineer_code: Annotated[list, add_messages]
    architect_messages: Annotated[list, add_messages]
    director_messages: Annotated[list, add_messages]
    qa_feedback: Annotated[list, add_messages]
    iteration_score: Annotated[list[int], operator.add]

    # Internal-only routing fields - never touched by the parent graph.
    syntax_ok: bool
    syntax_error: str

In [0]:
def syntax_checker_node(state: QAState):
    code_str = get_text(state["engineer_code"][-1])
    try:
        ast.parse(code_str)
        return {"syntax_ok": True, "syntax_error": ""}
    except SyntaxError as e:
        return {"syntax_ok": False, "syntax_error": f"{e.msg} (line {e.lineno})"}

In [0]:
def logic_tester_node(state: QAState):
    code_str = get_text(state["engineer_code"][-1])
    original_requirements = get_text(state["director_messages"][-1])
    design = get_text(state["architect_messages"][-1])

    system_prompt = (
        "You are a logic-focused QA engineer. The code has already been "
        "confirmed to be syntactically valid Python — do not comment on "
        "syntax. Your job is purely functional: compare the code against "
        "the original requirements and design, and list concrete logic "
        "bugs or missing features. Explicitly check for: flying obstacles, "
        "ground obstacles, jump/fall physics, a distinct duck mechanic, and "
        "a working high-score tracker. Do NOT write corrected code — "
        "diagnosis only. If the code is functionally correct, say so "
        "explicitly and report fewer than 2 issues rather than inventing any."
    )

    instruction = (
        f"Original requirements:\n{original_requirements}\n\n"
        f"Design:\n{design}\n\n"
        f"Code:\n{code_str}"
    )

    response = llm.invoke([
        ("system", system_prompt),
        ("human", instruction)
    ])

    return {"qa_feedback": [AIMessage(content=get_text(response))]}

In [0]:
def performance_auditor_node(state: QAState):
    # Path A: syntax failed upstream - score without an LLM call.
    if not state["syntax_ok"]:
        note = f"Code failed to parse: {state['syntax_error']}"
        return {
            "qa_feedback": [AIMessage(content=note)],
            "iteration_score": [1]
        }

    # Path B: syntax passed, logic_tester already appended its report.
    qa_report = get_text(state["qa_feedback"][-1])

    system_prompt = (
        "You are a scoring engine. Read the QA report and output ONLY "
        "a single integer from 1 to 10 representing overall code health "
        "and feature completeness. No words, no explanation, just the number.\n\n"
        "Scoring bands:\n"
        "9-10: No meaningful bugs remain, all requirements are met.\n"
        "7-8: Minor cosmetic or edge-case issues only.\n"
        "4-6: At least one functional bug, but the game is playable.\n"
        "1-3: Missing a core required feature (jumping, ducking, obstacles, or scoring).\n\n"
        "Do not default to the middle of the range — use the full scale."
    )

    response = llm.invoke([
        ("system", system_prompt),
        ("human", qa_report)
    ])

    raw = get_text(response).strip()
    match = re.search(r"\d+", raw)
    score = int(match.group()) if match else 5
    score = max(1, min(10, score))

    return {"iteration_score": [score]}

### Wiring and compiling the subgraph
Routing: `syntax_checker` → if syntax failed, skip straight to
`performance_auditor` (no wasted LLM call); if syntax passed, go through
`logic_tester` first.

In [0]:
qa_subgraph_builder = StateGraph(QAState)

qa_subgraph_builder.add_node("syntax_checker", syntax_checker_node)
qa_subgraph_builder.add_node("logic_tester", logic_tester_node)
qa_subgraph_builder.add_node("performance_auditor", performance_auditor_node)

qa_subgraph_builder.set_entry_point("syntax_checker")

def route_after_syntax(state: QAState):
    return "logic_tester" if state["syntax_ok"] else "performance_auditor"

qa_subgraph_builder.add_conditional_edges(
    "syntax_checker",
    route_after_syntax,
    {
        "logic_tester": "logic_tester",
        "performance_auditor": "performance_auditor"
    }
)

qa_subgraph_builder.add_edge("logic_tester", "performance_auditor")
qa_subgraph_builder.add_edge("performance_auditor", END)

# No separate checkpointer - it shares the parent's automatically when
# added as a node below. A separate MemorySaver here would fork its
# state history away from the parent run.
qa_subgraph = qa_subgraph_builder.compile()

## Building the Main Graph Structure

Same as the flat version, except the `"qa"` node is now the compiled
`qa_subgraph` object itself, not a plain function — LangGraph calls it
exactly like any other node, and the parent graph never sees the three
inner steps (`syntax_checker`, `logic_tester`, `performance_auditor`).

In [0]:
graph = StateGraph(GameState)

graph.add_node("director", director_node)
graph.add_node("architect", architect_node)
graph.add_node("engineer", engineer_node)
graph.add_node("save_code", save_code_node)
graph.add_node("execution", execution_node)
graph.add_node("qa", qa_subgraph)   # <- the whole subgraph, added as ONE node

graph.set_entry_point("director")

graph.add_edge("director", "architect")
graph.add_edge("architect", "engineer")
graph.add_edge("engineer", "save_code")
graph.add_edge("save_code", "execution")
graph.add_edge("execution", "qa")
# no separate "scorer" node/edge - the subgraph already produces the
# final score internally before returning control to the parent

def route_after_scoring(state: GameState):
    """
    Decides what happens after scoring: ask the human, then return the
    NAME of the next node to go to. Reads from iteration_score, which
    the qa_subgraph already populated internally.
    """
    latest_score = state["iteration_score"][-1]

    decision = interrupt(
        {"question": f"Current score: {latest_score}/10. Loop again for fixes, or finish?",
         "options": ["loop", "finish"]}
    )

    if decision == "finish":
        return "end"
    else:
        return "engineer"

graph.add_conditional_edges(
    "qa",
    route_after_scoring,
    {
        "engineer": "engineer",
        "end": END
    }
)

memory = MemorySaver()
app = graph.compile(checkpointer=memory)

## Running the Graph

In [0]:
thread_id = str(uuid.uuid4())
config = {"configurable": {"thread_id": thread_id}}

initial_state = {
    "director_messages": [],
    "architect_messages": [],
    "engineer_code": [],
    "qa_feedback": [],
    "current_actor": "director",
    "iteration": 0,
    "iteration_score": [],
    "file_saved": False
}

In [0]:
def run_graph_with_interrupts(app, initial_state, config):
    """
    Runs the graph, and every time it pauses on an interrupt(), prompts
    the human for input and resumes - repeating until the graph reaches
    END with no more pauses left.
    """
    stream_input = initial_state

    while True:
        interrupted = False

        for event in app.stream(stream_input, config=config):
            print(event)

            if "__interrupt__" in event:
                interrupted = True
                interrupt_data = event["__interrupt__"][0].value
                print("PAUSED:", interrupt_data)

                user_input = input("Your response: ")
                stream_input = Command(resume=user_input)

        if not interrupted:
            break

In [0]:
run_graph_with_interrupts(app, initial_state, config)

{'director': {'director_messages': [HumanMessage(content="Build a Dino Runner game (Chrome Dino style) in Python using pygame. It must include ALL of the following, exactly:\n1. Flying obstacles (e.g. Pterodactyls) at variable heights.\n2. Ground obstacles (e.g. Cacti).\n3. Accurate jumping and falling physics (gravity, velocity, a landing check).\n4. Jumping mapped to one key and ducking mapped to a different key, with a visibly smaller hitbox/sprite while ducking.\n5. A functional high-score tracking system (persists the best run's score within the session and displays both current score and high score).", additional_kwargs={}, response_metadata={}, id='4a808ae7-3c03-4175-9cb2-b5c0e949d459')], 'current_actor': 'architect'}}
{'architect': {'architect_messages': [AIMessage(content='### System Components and Features\n\nTo build a Dino Runner game in Python using pygame, the following system components and features are required:\n\n#### 1. Game Classes\n* **Dino Class**: Represents the 

Your response:  y

{'execution': {'qa_feedback': [ToolMessage(content='Process ran for 8+ seconds without crashing (timed out as expected for a headless game loop with no window to close).\n\nPartial STDOUT:\n\n\nPartial STDERR:\n', id='83ed665e-43b8-4c62-8897-25a55677e7ee', tool_call_id='execution_manager')], 'current_actor': 'qa'}}
{'__interrupt__': (Interrupt(value={'question': 'Current score: 5/10. Loop again for fixes, or finish?', 'options': ['loop', 'finish']}, id='f5cb9be3fd884a948a082d70330089fc'),)}
PAUSED: {'question': 'Current score: 5/10. Loop again for fixes, or finish?', 'options': ['loop', 'finish']}


Your response:  loop

{'qa': {'engineer_code': [AIMessage(content='import pygame\nimport sys\nimport random\n\n# Initialize Pygame\npygame.init()\n\n# Define game classes\nclass Dino:\n    def __init__(self):\n        self.position = [100, 400]\n        self.velocity = [0, 0]\n        self.jump_velocity = 20\n        self.gravity = 1\n        self.hitbox = pygame.Rect(self.position[0], self.position[1], 50, 50)\n        self.sprite = pygame.Surface((50, 50))\n        self.sprite.fill((255, 0, 0))\n        self.score = 0\n        self.high_score = 0\n        self.ducking = False\n\nclass Obstacle:\n    def __init__(self):\n        self.position = [1000, 450]\n        self.velocity = [-10, 0]\n        self.hitbox = pygame.Rect(self.position[0], self.position[1], 50, 50)\n        self.sprite = pygame.Surface((50, 50))\n        self.sprite.fill((0, 255, 0))\n\nclass Pterodactyl(Obstacle):\n    def __init__(self):\n        super().__init__()\n        self.position = [1000, random.randint(100, 300)]\n        self

Your response:  y

{'execution': {'qa_feedback': [ToolMessage(content='Process ran for 8+ seconds without crashing (timed out as expected for a headless game loop with no window to close).\n\nPartial STDOUT:\n\n\nPartial STDERR:\n', id='b547566a-a1a2-4b01-b1c4-66bfbba2ba61', tool_call_id='execution_manager')], 'current_actor': 'qa'}}
{'__interrupt__': (Interrupt(value={'question': 'Current score: 8/10. Loop again for fixes, or finish?', 'options': ['loop', 'finish']}, id='4c69538d8e81b7a0a0629f6418780d06'),)}
PAUSED: {'question': 'Current score: 8/10. Loop again for fixes, or finish?', 'options': ['loop', 'finish']}


Your response:  finish

{'qa': {'engineer_code': [AIMessage(content='import pygame\nimport sys\nimport random\n\n# Initialize Pygame\npygame.init()\n\n# Define game classes\nclass Dino:\n    def __init__(self):\n        self.position = [100, 400]\n        self.velocity = [0, 0]\n        self.jump_velocity = 20\n        self.gravity = 1\n        self.hitbox = pygame.Rect(self.position[0], self.position[1], 50, 50)\n        self.sprite = pygame.Surface((50, 50))\n        self.sprite.fill((255, 0, 0))\n        self.score = 0\n        self.high_score = 0\n        self.ducking = False\n\nclass Obstacle:\n    def __init__(self):\n        self.position = [1000, 450]\n        self.velocity = [-10, 0]\n        self.hitbox = pygame.Rect(self.position[0], self.position[1], 50, 50)\n        self.sprite = pygame.Surface((50, 50))\n        self.sprite.fill((0, 255, 0))\n\nclass Pterodactyl(Obstacle):\n    def __init__(self):\n        super().__init__()\n        self.position = [1000, random.randint(100, 300)]\n        self

## What each node adds to state, precisely
- **Director** — writes `director_messages`. No LLM.
- **Architect** — reads `director_messages`, calls the LLM, writes `architect_messages`.
- **Engineer** — reads `architect_messages` (first pass) or `engineer_code` + `qa_feedback` (later passes), calls the LLM, writes `engineer_code`, bumps `iteration`.
- **Save code** — reads `engineer_code`, writes the `.py` file to disk, sets `file_saved`. No LLM.
- **Execution** — pauses (interrupt) for approval, runs the file as a subprocess (with a dummy SDL driver for headless environments), writes the raw output into `qa_feedback`. No LLM.
- **QA subgraph (`"qa"` node)** — internally: `syntax_checker` (no LLM) → `logic_tester` (LLM, skipped if syntax failed) → `performance_auditor` (LLM unless syntax failed, appends to `iteration_score`).
- **Human review** — the conditional edge after `"qa"`: pauses again, asks loop/finish, routes back to Engineer or to END.